# 05 — Modelling & SHAP Analysis

Trains three classifiers using scikit-learn Pipelines and explains predictions with SHAP.

**Skills used:** `.claude/skills/scikit-learn/SKILL.md`

**Input:** `data/features.parquet`  
**Output:** `visuals/roc_curves.png`, `visuals/shap_summary_plot.png`

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

from src.model import train_evaluate, plot_confusion_matrix, plot_roc_curve

sns.set_theme(style='whitegrid', context='notebook')
VISUALS = ROOT / 'visuals'
VISUALS.mkdir(exist_ok=True)

RANDOM_STATE = 42

---
## Load & prepare data

In [ ]:
from sklearn.model_selection import train_test_split

df = pd.read_parquet(ROOT / 'data' / 'features.parquet')
df = df.dropna(subset=['position_gained'])

print(f"Feature matrix: {df.shape}")
print(f"Target balance: {df['position_gained'].value_counts(normalize=True).round(3).to_dict()}")

NUMERIC_FEATURES = ['stop_lap_pct', 'gap_to_car_ahead', 'compound_hardness',
                    'team_avg_stop_time', 'prior_stops']
BINARY_FEATURES  = ['is_undercut_attempt']
CAT_FEATURES     = ['circuit_type']

FEATURE_COLS = NUMERIC_FEATURES + BINARY_FEATURES + CAT_FEATURES
TARGET       = 'position_gained'

X = df[FEATURE_COLS]
y = df[TARGET].astype(int)

# Stratified split — preserves class balance in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"\nTrain: {X_train.shape}  |  Test: {X_test.shape}")

---
## Build preprocessing pipeline

Following the scikit-learn skill: always use `Pipeline` and `ColumnTransformer` to prevent data leakage and ensure consistency between train and test transforms.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,     NUMERIC_FEATURES + BINARY_FEATURES),
    ('cat', categorical_transformer, CAT_FEATURES),
])

print("Preprocessor defined.")

---
## Model 0 — Baseline (majority class)

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
baseline_metrics = train_evaluate(baseline, X_train, X_test, y_train, y_test)
print("Baseline:", baseline_metrics)

---
## Model 1 — Logistic Regression

Interpretable coefficients provide a useful baseline above the majority class. Scaling is critical for Logistic Regression — handled inside the Pipeline.

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

lr_metrics = train_evaluate(lr_pipeline, X_train, X_test, y_train, y_test)
print("Logistic Regression:", lr_metrics)

---
## Model 2 — XGBoost

Captures non-linear interactions between features — e.g. the interaction between `stop_lap_pct` and `circuit_type` that a linear model cannot represent. Tree-based models don't require feature scaling, but we keep it in the pipeline for consistency.

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score

xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier',   XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

xgb_metrics = train_evaluate(xgb_pipeline, X_train, X_test, y_train, y_test)
print("XGBoost:", xgb_metrics)

# 5-fold CV to confirm generalisation
cv_scores = cross_val_score(xgb_pipeline, X_train, y_train, cv=5, scoring='roc_auc')
print(f"\nXGBoost 5-fold CV AUC-ROC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

---
## Results table

In [ ]:
results_table = pd.DataFrame([
    {'Model': 'Baseline (majority class)', **baseline_metrics},
    {'Model': 'Logistic Regression',       **lr_metrics},
    {'Model': 'XGBoost',                   **xgb_metrics},
]).set_index('Model')

display(results_table.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}'))

# PRD acceptance criterion
xgb_auc = xgb_metrics['auc_roc']
assert xgb_auc >= 0.72, f"XGBoost AUC-ROC {xgb_auc:.3f} is below the PRD target of 0.72"
print(f"\nPRD target met: XGBoost AUC-ROC = {xgb_auc:.3f} >= 0.72")

---
## Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_confusion_matrix(baseline,    X_test, y_test, 'Baseline',            ax=axes[0])
plot_confusion_matrix(lr_pipeline, X_test, y_test, 'Logistic Regression', ax=axes[1])
plot_confusion_matrix(xgb_pipeline,X_test, y_test, 'XGBoost',             ax=axes[2])

plt.tight_layout()
plt.show()

---
## ROC curves

In [ ]:
plot_roc_curve(
    models={
        'Baseline':            baseline,
        'Logistic Regression': lr_pipeline,
        'XGBoost':             xgb_pipeline,
    },
    X_test=X_test,
    y_test=y_test,
    save_path=VISUALS / 'roc_curves.png',
)

---
## SHAP Analysis — XGBoost

SHAP (SHapley Additive exPlanations) explains how each feature contributes to individual predictions. The summary plot shows global feature importance; waterfall plots show individual predictions.

In [ ]:
import shap

# Extract the fitted XGBoost classifier from the pipeline
xgb_clf = xgb_pipeline.named_steps['classifier']

# Transform X_test through the preprocessor to get the feature matrix SHAP expects
X_test_transformed = xgb_pipeline.named_steps['preprocessor'].transform(X_test)

# Get feature names after one-hot encoding
num_feature_names = NUMERIC_FEATURES + BINARY_FEATURES
cat_feature_names = list(
    xgb_pipeline.named_steps['preprocessor']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(CAT_FEATURES)
)
all_feature_names = num_feature_names + cat_feature_names

X_test_df = pd.DataFrame(X_test_transformed, columns=all_feature_names)

explainer   = shap.TreeExplainer(xgb_clf)
shap_values = explainer(X_test_df)

print(f"SHAP values shape: {shap_values.values.shape}")

### SHAP Summary Plot

Each point is a prediction. Position on x-axis = SHAP value (impact on model output). Colour = feature value (red = high, blue = low).

In [ ]:
shap.summary_plot(shap_values, X_test_df, show=False)
plt.tight_layout()
plt.savefig(VISUALS / 'shap_summary_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: shap_summary_plot.png")

### SHAP Waterfall — Correct Prediction (True Positive)

In [ ]:
y_pred_xgb = xgb_pipeline.predict(X_test)
y_test_arr = y_test.values

# Find a true positive (predicted gain, actually gained)
tp_indices = np.where((y_pred_xgb == 1) & (y_test_arr == 1))[0]
tp_idx = tp_indices[0]

shap.waterfall_plot(shap_values[tp_idx], show=False)
plt.title("Waterfall — True Positive (predicted gain, actual gain)")
plt.tight_layout()
plt.savefig(VISUALS / 'shap_waterfall_tp.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: shap_waterfall_tp.png")

### SHAP Waterfall — Incorrect Prediction (False Positive)

In [ ]:
# Find a false positive (predicted gain, actually no gain)
fp_indices = np.where((y_pred_xgb == 1) & (y_test_arr == 0))[0]
fp_idx = fp_indices[0]

shap.waterfall_plot(shap_values[fp_idx], show=False)
plt.title("Waterfall — False Positive (predicted gain, actual no gain)")
plt.tight_layout()
plt.savefig(VISUALS / 'shap_waterfall_fp.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: shap_waterfall_fp.png")

---
## Final summary

The most important features (per SHAP) align with what F1 strategists prioritise in practice: `stop_lap_pct` (timing the window correctly), `is_undercut_attempt` (pitting before your rival), and `gap_to_car_ahead` (track position at pit entry). The XGBoost model captures the non-linear interaction between these factors — for example, an undercut is only effective if the timing window is right, which a linear model cannot fully express.

In [ ]:
print("=" * 50)
print("FINAL RESULTS")
print("=" * 50)
print(results_table.to_string())
print(f"\nXGBoost AUC-ROC: {xgb_metrics['auc_roc']:.4f}")
print(f"PRD target (>= 0.72): {'PASSED' if xgb_metrics['auc_roc'] >= 0.72 else 'FAILED'}")